In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/merged_train.csv", parse_dates=["Date"])

print(df.shape)
df.head()

C:\Users\chait\AppData\Local\Temp\ipykernel_39472\1618994028.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/merged_train.csv", parse_dates=["Date"])


(1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [3]:

df["CompetitionDistance"].fillna(df["CompetitionDistance"].median(), inplace=True)


df["CompetitionOpenSinceMonth"].fillna(0, inplace=True)
df["CompetitionOpenSinceYear"].fillna(0, inplace=True)


df["Promo2SinceWeek"].fillna(0, inplace=True)
df["Promo2SinceYear"].fillna(0, inplace=True)
df["PromoInterval"].fillna("None", inplace=True)


C:\Users\chait\AppData\Local\Temp\ipykernel_39472\3763639221.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["CompetitionDistance"].fillna(df["CompetitionDistance"].median(), inplace=True)
C:\Users\chait\AppData\Local\Temp\ipykernel_39472\3763639221.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values 

In [4]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfYear"] = df["Date"].dt.dayofyear
df["IsWeekend"] = df["Date"].dt.weekday.isin([5,6]).astype(int)


In [5]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ["StoreType", "Assortment", "StateHoliday", "PromoInterval"]
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))


In [6]:
df = df.sort_values(["Store", "Date"])

# Lag 7 days
df["Sales_lag_7"] = df.groupby("Store")["Sales"].shift(7)

# Lag 30 days
df["Sales_lag_30"] = df.groupby("Store")["Sales"].shift(30)


In [7]:
df["Sales_roll_mean_7"] = df.groupby("Store")["Sales"].shift(1).rolling(7).mean()
df["Sales_roll_mean_30"] = df.groupby("Store")["Sales"].shift(1).rolling(30).mean()


In [8]:
lag_cols = ["Sales_lag_7", "Sales_lag_30", "Sales_roll_mean_7", "Sales_roll_mean_30"]
df[lag_cols] = df[lag_cols].fillna(0)


In [9]:
df.to_csv("../data/train_processed.csv", index=False)
print("Feature engineering complete! Processed data saved to train_processed.csv")


Feature engineering complete! Processed data saved to train_processed.csv


In [10]:
df[["Sales", "Sales_lag_7", "Sales_roll_mean_7"]].head(10)


,Sales,Sales_lag_7,Sales_roll_mean_7
1016095,0,0.0,0.000000
1014980,5530,0.0,0.000000
1013865,4327,0.0,0.000000
1012750,4486,0.0,0.000000
1011635,4997,0.0,0.000000
1010520,0,0.0,0.000000
1009405,7176,0.0,0.000000
1008290,5580,0.0,3788.000000
1007175,5471,5530.0,4585.142857
1006060,4892,4327.0,4576.714286
